# `evaluate.py` — Batch Evaluation Script

## Purpose
Runs **batch evaluation** of the trained model on the full test (or val) split and writes structured results to the `evaluation/` directory.

## Difference from `inference.py`
| | `evaluate.py` | `inference.py` |
|-|-|-|
| Input | Full CSV split (batch) | Single review text |
| Output | JSON metrics + confusion matrices + CSV predictions | Single prediction dict |
| Use-case | Reproducing paper results, comparing checkpoints | Website API, interactive XAI |

## Usage
```bash
# From ml-research root:
python outputs/cosmetic_sentiment_v1/evaluation/evaluate.py \
    --checkpoint outputs/cosmetic_sentiment_v1/best_model.pt \
    --split test \
    --save_dir outputs/cosmetic_sentiment_v1/evaluation
```

## Output Files
| File | Description |
|------|-------------|
| `metrics.json` | All per-aspect and overall Macro-F1, Accuracy, MCC etc. |
| `predictions.csv` | Full prediction table (text, aspect, true_label, pred_label) |
| `all_confusion_matrices.png` | Grid of 7 confusion matrices (one per aspect) |
| `error_analysis.csv` | Misclassified samples for manual inspection |
| `mixed_sentiment_analysis.json` | MSR-specific metrics (mixed_detection_rate etc.) |

In [1]:
print("⏳ Starting: Running MSR evaluation...")
import time
_start_time = time.time()
import os, yaml, torch, argparse, sys
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm

ML_RESEARCH = os.path.dirname(os.path.abspath(''))  # ml-research/
SRC_DIR = os.path.join(ML_RESEARCH, 'src')
EVAL_DIR = os.path.join(ML_RESEARCH, 'outputs', 'cosmetic_sentiment_v1', 'evaluation')
for _p in tqdm([SRC_DIR, EVAL_DIR], desc="Processing _p"):
    if _p not in sys.path: sys.path.insert(0, _p)

from models.model import create_model
from utils.data_utils  import (DependencyParser, DependencyParsingDataset,
                                CosmeticReviewDataset, collate_fn_with_dependencies)
from utils.metrics     import AspectSentimentEvaluator, MixedSentimentEvaluator
from transformers      import RobertaTokenizer
from torch.utils.data  import DataLoader
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Running MSR evaluation.")


⏳ Starting: Running MSR evaluation...


Processing _p: 100%|██████████| 2/2 [00:00<?, ?it/s]


🕒 Done in 6.54s
✅ Completed: Running MSR evaluation.


## Loading the Checkpoint

In [2]:
print("⏳ Starting: Defining function load_model...")
import time
_start_time = time.time()
def load_model(checkpoint_path, device='cuda'):
    """
    Load a trained model from a checkpoint file.
    
    The checkpoint is a dict saved by ExperimentTrainer containing:
      - 'model_state_dict': the trained weights
      - 'config': the full training+model config (so model can be recreated without a separate YAML)
      - 'best_val_metric': the best val Macro-F1 achieved during training
    """
    print("📥 Loading a saved model checkpoint...")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    config     = checkpoint['config']

    print("🧩 Building the neural network architecture...")
    model = create_model(config)                              # Rebuild model architecture
    model.load_state_dict(checkpoint['model_state_dict'])    # Load trained weights
    model.to(device).eval()                                  # GPU + eval mode (disable dropout)

    best_metric = checkpoint.get('best_val_metric', 0.0)     # Use .get() to handle old checkpoints
    print(f'Model loaded from {checkpoint_path} — best_val_metric: {best_metric:.4f}')
    return model, config
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Defining function load_model.")


⏳ Starting: Defining function load_model...
🕒 Done in 0.00s
✅ Completed: Defining function load_model.


## Batch Prediction Loop

In [3]:
print("⏳ Starting: Defining function evaluate_model...")
from tqdm.auto import tqdm
import time
_start_time = time.time()
def evaluate_model(model, dataloader, config, device, save_dir=None):
    """
    Run the model over all batches in the dataloader and collect predictions.
    Computes per-aspect Macro-F1, overall metrics, MSR analysis, and (optionally) saves outputs.

    Args:
        model:       Trained MultiAspectSentimentModel (or baseline)
        dataloader:  DataLoader for the test or val split
        config:      Full config dict (aspects, model, data settings)
        device:      'cuda' or 'cpu'
        save_dir:    Optional Path to write results (metrics.json, predictions.csv, etc.)

    Returns:
        dict with 'overall', 'per_aspect', and 'mixed_sentiment' metrics
    """
    model.eval()
    all_predictions, all_labels, all_aspects, all_texts, all_review_ids = [], [], [], [], []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Evaluating'):
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            aspect_ids     = batch['aspect_ids'].to(device)
            labels         = batch['labels']

            # Pass edge_indices if the model uses GCN (None otherwise to save compute)
            edge_indices = None
            if config['model'].get('use_dependency_gcn', False):
                edge_indices = [e.to(device) if e is not None else None
                                for e in batch['edge_indices']]

            preds = model(input_ids, attention_mask, aspect_ids, edge_indices)
            if isinstance(preds, tuple): preds = preds[0]  # Unpack (logits, attn, repr) tuple

            pred_classes = torch.argmax(preds, dim=1).cpu().numpy()
            all_predictions.extend(pred_classes)
            all_labels.extend(labels.numpy())
            all_aspects.extend(batch['aspects'])
            all_texts.extend(batch['texts'])
            all_review_ids.extend(batch['review_ids'])

    # ── Compute metrics ─────────────────────────────────────────────
    evaluator      = AspectSentimentEvaluator(config['aspects']['names'])
    msr_evaluator  = MixedSentimentEvaluator(config['aspects']['names'])
    aspect_metrics = {}

    for aspect in config['aspects']['names']:
        mask  = np.array([a == aspect for a in all_aspects])
        if mask.sum() == 0: continue
        aspect_metrics[aspect] = evaluator.evaluate_aspect(
            np.array(all_labels)[mask],
            np.array(all_predictions)[mask],
            aspect
        )
        evaluator.print_results(aspect)

    overall = evaluator.evaluate_aspect(np.array(all_labels), np.array(all_predictions), 'overall')

    # ── MSR Analysis ────────────────────────────
    # Group true and predicted labels by review_id for the MSR evaluator
    review_true, review_pred = {}, {}
    for rid, asp, true_lbl, pred_lbl in zip(all_review_ids, all_aspects, all_labels, all_predictions):
        review_true.setdefault(rid, {})[asp] = int(true_lbl)
        review_pred.setdefault(rid, {})[asp] = int(pred_lbl)

    mixed_metrics = msr_evaluator.evaluate_mixed_sentiment_resolution(review_true, review_pred)
    msr_evaluator.print_mixed_sentiment_results(mixed_metrics)

    results = {
        'overall'         : {k: float(v) for k, v in overall.items() if not isinstance(v, (list, np.ndarray))},
        'per_aspect'      : {asp: {k: (v.tolist() if isinstance(v, np.ndarray) else v)
                                    for k, v in mets.items()}
                             for asp, mets in aspect_metrics.items()},
        'mixed_sentiment' : mixed_metrics,
    }

    # ── Save outputs ───────────────────────────
    if save_dir is not None:
        import json, matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt
        import seaborn as sns

        save_dir = Path(save_dir); save_dir.mkdir(parents=True, exist_ok=True)

        # metrics.json
        print("📝 Writing the final report to disk...")
        (save_dir / 'metrics.json').write_text(
            json.dumps(results, indent=2, default=str), encoding='utf-8'
        )

        # predictions.csv — full table of raw predictions for error analysis
        label_names = {0: 'negative', 1: 'neutral', 2: 'positive'}
        print("💾 Saving progress to a new CSV file...")
        pd.DataFrame({
            'text'      : all_texts,
            'aspect'    : all_aspects,
            'review_id' : all_review_ids,
            'true_label': [label_names.get(l, l) for l in all_labels],
            'pred_label': [label_names.get(p, p) for p in all_predictions],
            'correct'   : [t == p for t, p in zip(all_labels, all_predictions)],
        }).to_csv(save_dir / 'predictions.csv', index=False)

        # error_analysis.csv — only misclassified samples
        print("📖 Reading the dataset from CSV...")
        pred_df = pd.read_csv(save_dir / 'predictions.csv')
        pred_df[pred_df['correct'] == False].to_csv(save_dir / 'error_analysis.csv', index=False)

        # Confusion matrix grid
        aspects = config['aspects']['names']
        n_cols  = 4
        n_rows  = (len(aspects) + n_cols - 1) // n_cols
        print("📈 Preparing some visualizations to show you the results.")
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
        axes_flat = axes.flatten()
        for i, asp in enumerate(aspects):
            if asp in results['per_aspect']:
                cm = np.array(results['per_aspect'][asp]['confusion_matrix'])
                print("📉 Plotting detailed statistical distributions.")
                sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes_flat[i],
                           xticklabels=['neg', 'neu', 'pos'], yticklabels=['neg', 'neu', 'pos'])
                axes_flat[i].set_title(f'{asp} — Macro-F1: {results["per_aspect"][asp]["macro_f1"]:.3f}',
                                       fontweight='bold')
        for j in tqdm(range(len(aspects), len(axes_flat)), desc="J progress"): axes_flat[j].set_visible(False)
        plt.suptitle('Confusion Matrices per Aspect', fontsize=14, fontweight='bold')
        plt.tight_layout(rect=[0, 0, 1, 0.97])
        plt.savefig(save_dir / 'all_confusion_matrices.png', dpi=150, bbox_inches='tight')
        plt.close()

        # mixed_sentiment_analysis.json
        (save_dir / 'mixed_sentiment_analysis.json').write_text(
            json.dumps(mixed_metrics, indent=2), encoding='utf-8'
        )

        print(f'\nResults saved to: {save_dir}')

    return results
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Defining function evaluate_model.")


⏳ Starting: Defining function evaluate_model...
🕒 Done in 0.00s
✅ Completed: Defining function evaluate_model.


## CLI Entry Point

In [4]:
print("⏳ Starting: Loading dependencies and libraries...")
import time
_start_time = time.time()
# ── Run from CLI or modify the paths below to run interactively ──
# python outputs/cosmetic_sentiment_v1/evaluation/evaluate.py \
#     --checkpoint outputs/cosmetic_sentiment_v1/best_model.pt \
#     --split test \
#     --save_dir outputs/cosmetic_sentiment_v1/evaluation

# ── or interactively: ──
# ckpt    = 'outputs/cosmetic_sentiment_v1/best_model.pt'
# model, config = load_model(ckpt)
# tokenizer = RobertaTokenizer.from_pretrained(config['model']['roberta_model'])
# dataset   = CosmeticReviewDataset(config['data']['test_path'], tokenizer, config, config['aspects']['names'])
# loader    = DataLoader(dataset, batch_size=16, shuffle=False, collate_fn=collate_fn_with_dependencies)
# results   = evaluate_model(model, loader, config, device='cuda', save_dir='outputs/cosmetic_sentiment_v1/evaluation')

print('evaluate.py loaded. Use CLI or run interactively via the cells above.')
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Loading dependencies and libraries.")


⏳ Starting: Loading dependencies and libraries...
evaluate.py loaded. Use CLI or run interactively via the cells above.
🕒 Done in 0.00s
✅ Completed: Loading dependencies and libraries.
